# AnnoIndex Pipeline

This notebook converts the AnnoIndex project into a standalone pipeline using Gemini LLMs. It will download the benchmark dataset (e.g. from the QUEST repository) and execute schema induction, indexing, and querying.

## 1. Setup Environment and Dependencies

In [ ]:

# Install necessary libraries including Google GenAI
!pip install -q pandas numpy openai google-generativeai

import os
import re
import json
import time
import random
from typing import List, Dict, Optional
from datetime import datetime
import pandas as pd

# Load Gemini API Key from Colab Secrets
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    print("Successfully loaded GEMINI_API_KEY from Colab secrets.")
except Exception as e:
    print("Not running in Colab or GEMINI_API_KEY secret not found.")
    # Fallback to getpass if running locally or secret is missing
    import getpass
    GEMINI_API_KEY = getpass.getpass("Enter your Gemini API Key: ")

import google.generativeai as genai
genai.configure(api_key=GEMINI_API_KEY)


## 2. Download Datasets

We download the required dataset files directly from the QUEST GitHub repository if they do not already exist.

In [ ]:

# Download datasets from the QUEST repository
import subprocess
import os

DATA_ROOT = "./data"
REPO_URL = "https://github.com/qiyandeng/QUEST.git"

if not os.path.exists(DATA_ROOT):
    print("Data directory not found. Downloading from QUEST repository...")
    # Clone the repository sparsely to just get the data directory
    commands = [
        "git clone --depth 1 --filter=blob:none --sparse " + REPO_URL + " temp_quest",
        "cd temp_quest && git sparse-checkout set data",
        "mv temp_quest/data ./data",
        "rm -rf temp_quest"
    ]
    for cmd in commands:
        subprocess.run(cmd, shell=True, check=True)
    print("Download complete.")
else:
    print("Data directory already exists. Skipping download.")

# Select which datasets to run
# Available options in QUEST data folder usually include 'lcr', 'swde', 'wikitext'
DATASETS_TO_RUN = ['lcr', 'swde', 'wikitext'] 


## 3. Configuration

In [ ]:

# config.py
class Config:
    # ---------- Paths ----------
    DATA_ROOT = "./data"
    RAW_DIR = os.path.join(DATA_ROOT, "raw")
    GROUND_TRUTH_DIR = os.path.join(DATA_ROOT, "ground_truth")
    QUERY_DIR = os.path.join(DATA_ROOT, "queries")
    INDEX_DIR = "./index_storage"
    RESULT_DIR = "./results"

    # ---------- SchemaLoop ----------
    SR_THRESHOLD = 0.6          # extraction success rate
    FE_THRESHOLD = 0.3          # filtering efficiency
    MAX_SCHEMA_ITER = 5

    # ---------- Query Engine ----------
    LLM_BUDGET = 10             # max LLM calls per query (soft limit)
    REUSE_THRESHOLD = 10        # promote virtual field after this many extractions
    EXTRACT_ORDER = ["regex", "slm", "llm"]   # ascending cost

    # ---------- Evaluation ----------
    SAMPLE_SIZE_FOR_VERIFICATION = 5


## 4. Utilities and Formatting

Helper functions to normalize extracted attributes and evaluate boolean operations in queries.

In [ ]:

# utils.py
class Normalizer:
    @staticmethod
    def normalize_value(value, attr_type="string"):
        if attr_type == "date":
            return Normalizer._normalize_date(value)
        elif attr_type == "numeric":
            return Normalizer._normalize_numeric(value)
        else:
            return Normalizer._normalize_string(value)

    @staticmethod
    def _normalize_date(value):
        if isinstance(value, str):
            m = re.match(r"\d{4}-\d{2}-\d{2}", value)
            if m:
                return value
            try:
                dt = datetime.strptime(value, "%Y-%m-%d")
                return dt.isoformat()
            except:
                pass
        return value

    @staticmethod
    def _normalize_numeric(value):
        if isinstance(value, (int, float)):
            return value
        if isinstance(value, str):
            cleaned = re.sub(r'[^\d.]', '', value)
            try:
                return float(cleaned)
            except:
                pass
        return value

    @staticmethod
    def _normalize_string(value):
        if isinstance(value, str):
            return value.strip().lower()
        return value


# cal_sel.py (Extract Filter Conditions)
def extract_filter_condition(where_clause):
    conditions = []
    current_condition = []
    paren_stack = 0
    tokens = re.split(r"\s+AND\s+|\s+OR\s+|\(|\)", where_clause, flags=re.IGNORECASE)
    for token in tokens:
        token = token.strip()
        if token == '':
            continue
        if token == '(':
            paren_stack += 1
            if current_condition:
                conditions.append(' '.join(current_condition).strip())
                current_condition = []
            current_condition.append(token)
        elif token == ')':
            paren_stack -= 1
            current_condition.append(token)
            if paren_stack == 0:
                conditions.append(' '.join(current_condition).strip())
                current_condition = []
        elif token.upper() == 'AND' or token.upper() == 'OR':
            if paren_stack == 0 and current_condition:
                conditions.append(' '.join(current_condition).strip())
                current_condition = []
            elif paren_stack > 0:
                current_condition.append(token.upper())
        else:
            current_condition.append(token)
            if paren_stack == 0 and (len(current_condition) == 3 or len(current_condition) > 3 and (current_condition[-2].upper() in ['AND', 'OR'])):
                conditions.append(' '.join(current_condition).strip())
                current_condition = []
    if current_condition:
        conditions.append(' '.join(current_condition).strip())
    final_conditions = []
    for condition in conditions:
        if condition.startswith('(') and condition.endswith(')'):
            inner_conditions = extract_filter_condition(condition[1:-1].strip())
            final_conditions.extend(inner_conditions)
        else:
            final_conditions.append(condition)
    return final_conditions

# util_order.py (Parse SQL)
def parse_sql(sql_query):
    # Extract WHERE clause
    match = re.search(r'WHERE (.+)', sql_query, re.IGNORECASE)
    if match:
        return match.group(1)
    return ""


## 5. LLM Integration (Gemini)

We adapt the OpenAI wrapper to instead use Google Gemini via `google-generativeai`.

In [ ]:

# llm.py adapted for Gemini
# We'll use gemini-1.5-flash for rapid schema extraction and query extraction
MODEL_NAME = "gemini-1.5-flash"

def ask_completion_with_retry(prompt_text, max_tokens=1000, temperature=0, retries=3, delay=5):
    attempt = 0
    while attempt < retries:
        try:
            model = genai.GenerativeModel(MODEL_NAME, generation_config=genai.GenerationConfig(
                temperature=temperature,
                max_output_tokens=max_tokens
            ))
            response = model.generate_content(prompt_text)
            if response.text:
                return response.text.strip()
            else:
                raise ValueError("Empty response text.")
        except Exception as e:
            print(f"LLM error: {e}. Retrying in {delay}s...")
            attempt += 1
            time.sleep(delay)
    raise Exception(f"Failed after {retries} retries.")

def ask_extract_attribute(attribute, text):
    prompt = f"""
    Extract the value of '{attribute}' from the following text.
    Return only the value, nothing else.
    If it is not present or cannot be determined, return an empty string.
    
    Text: {text[:2000]}
    """
    try:
        return ask_completion_with_retry(prompt, max_tokens=50, temperature=0)
    except Exception as e:
        print(f"Extraction failed for {attribute}: {e}")
        return ""


## 6. Data Loading

In [ ]:

# data_loader.py
class DataLoader:
    @staticmethod
    def load_documents(dataset_name: str) -> List[Dict]:
        doc_dir = os.path.join(Config.RAW_DIR, dataset_name)
        if not os.path.exists(doc_dir):
            print(f"Warning: Document directory not found: {doc_dir}")
            return []
        docs = []
        for filename in os.listdir(doc_dir):
            if filename.endswith(".txt"):
                doc_id = filename[:-4]   # strip .txt
                with open(os.path.join(doc_dir, filename), "r", encoding="utf-8", errors="ignore") as f:
                    content = f.read()
                docs.append({"doc_id": doc_id, "content": content})
        return docs

    @staticmethod
    def load_ground_truth(dataset_name: str) -> Optional[pd.DataFrame]:
        gt_path = os.path.join(Config.GROUND_TRUTH_DIR, f"{dataset_name}.csv")
        if os.path.exists(gt_path):
            return pd.read_csv(gt_path)
        return None

    @staticmethod
    def load_queries(dataset_name: str) -> List[str]:
        q_path = os.path.join(Config.QUERY_DIR, f"{dataset_name}.txt")
        if not os.path.exists(q_path):
            print(f"Warning: Query file not found: {q_path}")
            return []
        with open(q_path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip() and not line.startswith("...") and not line.startswith("--")]
        return lines


## 7. Schema Induction

In [ ]:

# schema_induction.py
class SchemaLoop:
    def __init__(self):
        self.dataset_schema = {}
        self.table_schemas = {}
        self.row_schemas = {}

    def induce(self, documents: List[Dict]) -> Dict:
        if not documents:
            return {"dataset_schema": {}, "table_schemas": {}, "row_schemas": {}}
            
        dataset_type = self._detect_dataset(documents)
        self.dataset_schema = {"domain": dataset_type, "source": "real"}

        groups = {"all": documents}
        table_schemas = {}
        for group_name, docs in groups.items():
            summary = self._summarize(docs)
            schema = self._generate_table_schema(summary)
            schema = self._verify_and_refine(schema, docs, level="table")
            table_schemas[group_name] = schema
        self.table_schemas = table_schemas

        row_schemas = {}
        for table_name, table_attrs in table_schemas.items():
            refined = self._generate_row_schema(table_attrs, documents)
            refined = self._verify_and_refine(refined, documents, level="row")
            row_schemas[table_name] = refined
        self.row_schemas = row_schemas

        return {
            "dataset_schema": self.dataset_schema,
            "table_schemas": self.table_schemas,
            "row_schemas": self.row_schemas
        }

    def _detect_dataset(self, docs):
        text = docs[0]['content'].lower()
        if "case" in text and "court" in text:
            return "lcr"
        elif "player" in text and "team" in text:
            return "wikitext"
        else:
            return "swde"

    def _summarize(self, docs):
        return " ".join([d['content'][:300] for d in docs[:3]])

    def _generate_table_schema(self, summary):
        prompt = f"From the following document summary, list the most common attributes (fields) that appear. Return only attribute names separated by commas.\nSummary: {summary}"
        response = ask_completion_with_retry(prompt, max_tokens=200, temperature=0)
        attrs = [a.strip() for a in response.split(",") if a.strip()]
        return attrs

    def _generate_row_schema(self, table_attrs, docs):
        return table_attrs

    def _verify_and_refine(self, schema, docs, level="row", max_iter=Config.MAX_SCHEMA_ITER):
        current = schema
        for _ in range(max_iter):
            sr, fe = self._compute_metrics(current, docs)
            if sr >= Config.SR_THRESHOLD and fe >= Config.FE_THRESHOLD:
                break
            prompt = f"Current schema: {current}. Extraction success rate = {sr:.2f}, filtering efficiency = {fe:.2f}. Suggest improvements (merge/split/rename). Return new attribute list separated by commas."
            response = ask_completion_with_retry(prompt, max_tokens=200, temperature=0)
            current = [a.strip() for a in response.split(",") if a.strip()]
        return current

    def _compute_metrics(self, schema, docs):
        sample = random.sample(docs, min(Config.SAMPLE_SIZE_FOR_VERIFICATION, len(docs)))
        total_pairs = len(sample) * len(schema)
        success = 0
        value_vectors = []
        for doc in sample:
            row = []
            for attr in schema:
                val = self._mock_extract(doc['content'], attr)
                if val:
                    success += 1
                    row.append(val)
                else:
                    row.append("")
            value_vectors.append(tuple(row))
        sr = success / total_pairs if total_pairs > 0 else 0.0
        from collections import Counter
        counter = Counter(value_vectors)
        max_fraction = max(counter.values()) / len(value_vectors) if value_vectors else 0
        fe = 1 - max_fraction
        return sr, fe

    def _mock_extract(self, text, attr):
        patterns = {
            "court_name": r'court[:\s]+([^\s,]+)',
            "hearing_year": r"(\d{4})",
            "judgment_year": r"judgment[:\s]+(\d{4})",
            "verdict": r'verdict[:\s]+([^\s,]+)',
            "player_name": r'([A-Z][a-z]+ [A-Z][a-z]+)',
            "team_name": r'team[:\s]+([^\s,]+)',
            "birth_year": r"born in (\d{4})",
            "active_status": r"active[:\s]+(True|False)",
        }
        if attr in patterns:
            m = re.search(patterns[attr], text, re.IGNORECASE)
            return m.group(1) if m else ""
        return ""


## 8. Annotation Index

In [ ]:

# annotation_index.py
class AnnotationIndex:
    def __init__(self, storage_dir=Config.INDEX_DIR):
        self.storage_dir = storage_dir
        os.makedirs(storage_dir, exist_ok=True)
        self.documents = {}
        self.attributes = {}
        self.schemas = {}
        self.virtual_fields = {}
        self.field_count = {}

    def build(self, documents: List[Dict], schemas: Dict):
        self.schemas = schemas
        for doc in documents:
            doc_id = doc['doc_id']
            self.documents[doc_id] = doc['content']
            self.attributes[doc_id] = {}
            if 'attributes' in doc:
                self.attributes[doc_id] = doc['attributes']
        self.virtual_fields = {doc_id: {} for doc_id in self.documents}
        self._persist()

    def _persist(self):
        data = {
            "documents": self.documents,
            "attributes": self.attributes,
            "schemas": self.schemas,
            "virtual_fields": self.virtual_fields,
            "field_count": self.field_count
        }
        with open(os.path.join(self.storage_dir, "index.json"), "w") as f:
            json.dump(data, f)

    def filter(self, predicates: List[Dict]) -> List[str]:
        doc_ids = list(self.documents.keys())
        for pred in predicates:
            attr = pred['attribute']
            op = pred['operator']
            value = pred['value']
            new_ids = []
            for doc_id in doc_ids:
                val = self.attributes.get(doc_id, {}).get(attr)
                if val is None:
                    val = self.virtual_fields.get(doc_id, {}).get(attr)
                if val is not None:
                    if op == '=' and str(val) == str(value):
                        new_ids.append(doc_id)
                    elif op == '<' and float(val) < float(value):
                        new_ids.append(doc_id)
                    elif op == '>' and float(val) > float(value):
                        new_ids.append(doc_id)
                    elif op == '<=' and float(val) <= float(value):
                        new_ids.append(doc_id)
                    elif op == '>=' and float(val) >= float(value):
                        new_ids.append(doc_id)
                    elif op == '!=' and str(val) != str(value):
                        new_ids.append(doc_id)
            doc_ids = new_ids
        return doc_ids

    def get_document(self, doc_id):
        return self.documents.get(doc_id, "")

    def add_virtual_field(self, doc_id, field, value):
        if doc_id not in self.virtual_fields:
            self.virtual_fields[doc_id] = {}
        self.virtual_fields[doc_id][field] = value
        self.field_count[field] = self.field_count.get(field, 0) + 1
        self._persist()

    def promote_virtual_field(self, field):
        if self.field_count.get(field, 0) >= Config.REUSE_THRESHOLD:
            for table in self.schemas.get('row_schemas', {}):
                if field not in self.schemas['row_schemas'][table]:
                    self.schemas['row_schemas'][table].append(field)
            for doc_id, fields in self.virtual_fields.items():
                if field in fields:
                    self.attributes[doc_id][field] = fields[field]
                    del self.virtual_fields[doc_id][field]
            self._persist()
            return True
        return False


## 9. Query Engine

In [ ]:

# query_engine.py
class QueryEngine:
    def __init__(self, index: AnnotationIndex, schemas: Dict, sampled_data: pd.DataFrame):
        self.index = index
        self.schemas = schemas
        self.sampled_data = sampled_data
        self.llm_calls = 0

    def execute(self, sql_query: str, mode='performance') -> List[Dict]:
        where_clause = parse_sql(sql_query)
        filter_conditions = extract_filter_condition(where_clause)
        schema_conditions = []
        extract_conditions = []
        all_attrs = self._get_all_attrs()

        for cond in filter_conditions:
            parts = cond.split()
            if len(parts) >= 3:
                attr = parts[0]
                op = parts[1]
                val = parts[2]
                if attr in all_attrs:
                    schema_conditions.append({"attribute": attr, "operator": op, "value": val})
                else:
                    extract_conditions.append(cond)

        doc_ids = self.index.filter(schema_conditions)

        result_docs = []
        for doc_id in doc_ids:
            doc_text = self.index.get_document(doc_id)
            skip = False
            for cond in extract_conditions:
                parts = cond.split()
                if len(parts) < 3: 
                    continue
                attr = parts[0]
                expected_val = parts[2].strip("'\"")
                
                if mode == 'performance':
                    model = 'llm'
                else:
                    model = 'slm'
                    
                extracted = self._extract_attribute(doc_id, attr, model)
                if extracted is None or str(extracted).strip().lower() != expected_val.lower():
                    skip = True
                    break
                    
                self.index.add_virtual_field(doc_id, attr, extracted)
                self.index.promote_virtual_field(attr)
                
            if not skip:
                attrs = self.index.attributes.get(doc_id, {}).copy()
                virtual = self.index.virtual_fields.get(doc_id, {})
                attrs.update(virtual)
                result_docs.append({
                    "doc_id": doc_id,
                    "content": doc_text,
                    "attributes": attrs
                })
        return result_docs

    def _get_all_attrs(self):
        attrs = set()
        for table in self.schemas.get('row_schemas', {}):
            attrs.update(self.schemas['row_schemas'][table])
        return attrs

    def _extract_attribute(self, doc_id, attr, model):
        text = self.index.get_document(doc_id)
        if model == 'regex':
            pattern = rf'{attr}[:\s]+([^\s,]+)'
            m = re.search(pattern, text, re.IGNORECASE)
            return m.group(1) if m else None
        else:
            return self._call_llm_extract(text, attr)

    def _call_llm_extract(self, text, attr):
        self.llm_calls += 1
        return ask_extract_attribute(attr, text[:2000])


## 10. Evaluator

In [ ]:

# evaluator.py
class Evaluator:
    @staticmethod
    def compute_f1(predicted: List[Dict], ground_truth: pd.DataFrame) -> Dict:
        gt_dict = {}
        for _, row in ground_truth.iterrows():
            doc_id = row['doc_id']
            gt_dict[doc_id] = {str(k): str(v) for k, v in row.items() if k != 'doc_id' and pd.notna(v)}
        pred_dict = {d['doc_id']: {str(k): str(v) for k, v in d['attributes'].items()} for d in predicted}

        tp = fp = fn = 0
        for doc_id, pred_attrs in pred_dict.items():
            if doc_id in gt_dict:
                gt_attrs = gt_dict[doc_id]
                match = True
                for k, v in gt_attrs.items():
                    if k not in pred_attrs or pred_attrs[k].lower() != v.lower():
                        match = False
                        break
                if match:
                    tp += 1
                else:
                    fp += 1
            else:
                fp += 1
        for doc_id in gt_dict:
            if doc_id not in pred_dict:
                fn += 1

        precision = tp / (tp+fp) if tp+fp > 0 else 0
        recall = tp / (tp+fn) if tp+fn > 0 else 0
        f1 = 2 * precision * recall / (precision+recall) if precision+recall > 0 else 0
        return {"precision": precision, "recall": recall, "f1": f1}


## 11. Main Execution

In [ ]:

# main.py execution block
def run_experiment(dataset_name):
    print(f"\n=== Running AnnoIndex on {dataset_name} ===")

    docs = DataLoader.load_documents(dataset_name)
    print(f"  Loaded {len(docs)} documents")
    if not docs:
        print("  No documents found. Skipping.")
        return None

    gt_df = DataLoader.load_ground_truth(dataset_name)
    if gt_df is not None:
        print(f"  Loaded ground truth with {len(gt_df)} entries")
    else:
        print("  No ground truth found. Evaluation will be skipped.")

    schema_loop = SchemaLoop()
    schemas = schema_loop.induce(docs)
    print(f"  Induced row schemas: {schemas['row_schemas']}")

    index = AnnotationIndex()
    index.build(docs, schemas)
    print(f"  Built index with {len(index.documents)} documents")

    queries = DataLoader.load_queries(dataset_name)
    print(f"  Loaded {len(queries)} queries")

    sample_df = pd.DataFrame([{'doc_id': d['doc_id'], **d.get('attributes', {})} for d in docs[:10]])
    if gt_df is not None:
        sample_df = sample_df.merge(gt_df, on='doc_id', how='left')

    engine = QueryEngine(index, schemas, sample_df)

    all_predicted = []
    # Note: Using first 5 queries here to speed up demo on Colab.
    # Change to `queries` for the full benchmark.
    for i, q in enumerate(queries[:5]): 
        results = engine.execute(q, mode='performance')
        all_predicted.extend(results)
        print(f"  Query {i+1}: {q[:60]}... -> {len(results)} docs")

    if gt_df is not None:
        metrics = Evaluator.compute_f1(all_predicted, gt_df)
        print(f"  Precision: {metrics['precision']:.3f}, Recall: {metrics['recall']:.3f}, F1: {metrics['f1']:.3f}")
        print(f"  Total LLM calls: {engine.llm_calls}")
        return metrics
    else:
        print("  Evaluation skipped.")
        return None

# Run the pipeline
os.makedirs(Config.RESULT_DIR, exist_ok=True)

results = {}
for ds in DATASETS_TO_RUN:
    res = run_experiment(ds)
    if res:
        results[ds] = res

if results:
    print("\n=== Summary ===")
    for ds, m in results.items():
        print(f"{ds}: F1={m['f1']:.3f}")
    with open(os.path.join(Config.RESULT_DIR, "summary.json"), "w") as f:
        json.dump(results, f, indent=2)
